In [ ]:
!pip install tensorflow-model-optimization

In [ ]:
pip install tf_keras

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_model_optimization as tfmot
import numpy as np

# 1. Hyperparameters & Setup
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 37 # Oxford-IIIT Pet dataset has 37 classes

# 2. Load and Preprocess Dataset
def preprocess_image(features):
    image = tf.image.resize(features['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0 # Normalize to [0, 1]
    label = features['label']
    return image, label

print("Loading dataset...")
(ds_train, ds_val), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train', 'test'],
    with_info=True,
    as_supervised=False # Keep as dict to access 'image' and 'label'
)

train_dataset = ds_train.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = ds_val.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 3. Build Base Model
print("Building MobileNetV3 Small...")
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = True # Fine-tune the whole model

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

Loading dataset...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.Y6BBCG_4.0.0/oxford_iiit_pet-train.tfrecord*...…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.Y6BBCG_4.0.0/oxford_iiit_pet-test.tfrecord*...:…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.
Building MobileNetV3 Small...
4334752/4334752 [==============================] - 0s 0us/step


In [ ]:
def augment_image(image, label):
    # Randomly flip horizontally
    image = tf.image.random_flip_left_right(image)
    # Randomly adjust brightness and contrast
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    # Ensure values stay between [0, 1] after adjustments
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

# Apply augmentation ONLY to the training dataset
train_dataset = ds_train.map(preprocess_image) \
                        .map(augment_image) \
                        .batch(BATCH_SIZE) \
                        .prefetch(tf.data.AUTOTUNE)

# Validation stays the same (no augmentation)
val_dataset = ds_val.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 4. Apply Pruning
EPOCHS = 11
WARMUP_EPOCHS = 5
steps_per_epoch = np.ceil(ds_info.splits['train'].num_examples / BATCH_SIZE).astype(np.int32)

pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
          initial_sparsity=0.00, # Start at 0% to let weights stabilize
          final_sparsity=0.50,   # 50% is much safer for MobileNet
          begin_step=WARMUP_EPOCHS * steps_per_epoch, # Wait 5 epochs before pruning
          end_step=EPOCHS * steps_per_epoch
      )
}

print("Wrapping model for pruning...")

def apply_pruning(layer):
    if isinstance(layer, (tf.keras.layers.Dense, tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
        return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params)
    return layer

pruned_base_model = tf.keras.models.clone_model(
    base_model,
    clone_function=apply_pruning
)

pruned_model = tf.keras.Sequential([
    pruned_base_model,
    tfmot.sparsity.keras.prune_low_magnitude(
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
        **pruning_params
    )
])

pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks for pruning AND dynamic learning rate
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,      # Cut learning rate in half
        patience=3,      # If val_loss doesn't improve for 3 epochs
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True
    )
]

Wrapping model for pruning...
Training and pruning model...
Epoch 1/11
115/115 [==============================] - 67s 237ms/step - loss: 3.1938 - accuracy: 0.1935 - val_loss: 3.8946 - val_accuracy: 0.0134 - lr: 1.0000e-04
Epoch 2/11
115/115 [==============================] - 26s 227ms/step - loss: 1.6633 - accuracy: 0.5500 - val_loss: 3.9003 - val_accuracy: 0.0262 - lr: 1.0000e-04
Epoch 3/11
115/115 [==============================] - 28s 244ms/step - loss: 1.1025 - accuracy: 0.7095 - val_loss: 3.9497 - val_accuracy: 0.0264 - lr: 1.0000e-04
Epoch 4/11
115/115 [==============================] - ETA: 0s - loss: 0.7993 - accuracy: 0.7916
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
115/115 [==============================] - 26s 225ms/step - loss: 0.7993 - accuracy: 0.7916 - val_loss: 4.0003 - val_accuracy: 0.0264 - lr: 1.0000e-04
Epoch 5/11
115/115 [==============================] - 26s 226ms/step - loss: 0.6021 - accuracy: 0.8571 - val_loss: 4.0083 - val_acc

In [ ]:
# 5. Train the Pruned Model
print("Training and pruning model...")
pruned_model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, callbacks=callbacks)

# 6. Strip Pruning Wrappers (Crucial step before saving/conversion)
print("Stripping pruning wrappers...")
model_for_export = tfmot.sparsity.keras.strip_pruning(pruned_model)

# 7. INT8 Quantization (Post-Training)
print("Quantizing to INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset is required for full INT8 quantization
def representative_data_gen():
    for input_value, _ in train_dataset.take(100):
        yield [input_value]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  # Input is INT8
converter.inference_output_type = tf.int8 # Output is INT8

tflite_quant_model = converter.convert()

# 8. Save the final model for the Raspberry Pi
with open('mobilenet_v3_pet_pruned_quantized.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Success! Model saved as 'mobilenet_v3_pet_pruned_quantized.tflite'")

Training and pruning model...
Epoch 1/11
115/115 [==============================] - 27s 231ms/step - loss: 0.6107 - accuracy: 0.8611 - val_loss: 4.0691 - val_accuracy: 0.0273 - lr: 2.5000e-05
Epoch 2/11
115/115 [==============================] - 26s 226ms/step - loss: 0.6985 - accuracy: 0.8408 - val_loss: 3.8908 - val_accuracy: 0.0286 - lr: 2.5000e-05
Epoch 3/11
115/115 [==============================] - 26s 227ms/step - loss: 0.7440 - accuracy: 0.8008 - val_loss: 3.8139 - val_accuracy: 0.0264 - lr: 2.5000e-05
Epoch 4/11
115/115 [==============================] - 26s 224ms/step - loss: 0.7703 - accuracy: 0.7818 - val_loss: 3.8809 - val_accuracy: 0.0264 - lr: 2.5000e-05
Epoch 5/11
115/115 [==============================] - 26s 223ms/step - loss: 0.7102 - accuracy: 0.8065 - val_loss: 3.8294 - val_accuracy: 0.0264 - lr: 2.5000e-05
Epoch 6/11
115/115 [==============================] - ETA: 0s - loss: 0.6432 - accuracy: 0.8293
Epoch 6: ReduceLROnPlateau reducing learning rate to 1.249999968

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Success! Model saved as 'mobilenet_v3_pet_pruned_quantized.tflite'


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_model_optimization as tfmot

# --- 1. Hyperparameters & Setup ---
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 37

# --- 2. Load and Preprocess Dataset ---
def preprocess_image(features):
    image = tf.image.resize(features['image'], (IMG_SIZE, IMG_SIZE))
    # MobileNetV2 expects float inputs in the range [-1.0, 1.0]
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0
    label = features['label']
    return image, label

def augment_image(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label

print("Loading dataset...")
(ds_train, ds_val), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train', 'test'],
    with_info=True,
    as_supervised=False
)

train_dataset = ds_train.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE) \
                        .map(augment_image, num_parallel_calls=tf.data.AUTOTUNE) \
                        .batch(BATCH_SIZE) \
                        .prefetch(tf.data.AUTOTUNE)

val_dataset = ds_val.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE) \
                    .batch(BATCH_SIZE) \
                    .prefetch(tf.data.AUTOTUNE)

# --- 3. Build MobileNetV2 ---
print("Building MobileNetV2...")
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(base_model.output)
model = tf.keras.Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- 4. Train the Float Model ---
print("Training the classification head...")
model.fit(train_dataset, validation_data=val_dataset, epochs=5)

print("Unfreezing base model for fine-tuning...")
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)]
model.fit(train_dataset, validation_data=val_dataset, epochs=10, callbacks=callbacks)

# --- 5. Apply QAT (Natively Supported) ---
print("\nPreparing for Quantization-Aware Training (QAT)...")

# Because it's MobileNetV2, we can safely quantize the ENTIRE model.
# This ensures Batch Normalization layers are correctly folded into Conv2D layers.
qat_model = tfmot.quantization.keras.quantize_model(model)

qat_model.compile(
    # Use a low learning rate to gently calibrate the INT8 fake-quantization nodes
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning with QAT...")
# 2 epochs is usually enough to recover accuracy
qat_model.fit(train_dataset, validation_data=val_dataset, epochs=2)

# --- 6. INT8 Quantization ---
print("\nQuantizing to TFLite INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    # Provide the [-1, 1] scaled images for accurate activation calibration
    for input_value, _ in train_dataset.unbatch().batch(1).take(300):
        yield [input_value]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open('mobilenet_v2_pet_qat.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Success! QAT Model saved as 'mobilenet_v2_pet_qat.tflite'")

Loading dataset...
Building MobileNetV2...
9406464/9406464 [==============================] - 2s 0us/step
Training the classification head...
Epoch 1/5
115/115 [==============================] - 25s 188ms/step - loss: 1.0803 - accuracy: 0.7432 - val_loss: 0.4831 - val_accuracy: 0.8640
Epoch 2/5
115/115 [==============================] - 21s 180ms/step - loss: 0.2797 - accuracy: 0.9361 - val_loss: 0.3954 - val_accuracy: 0.8795
Epoch 3/5
115/115 [==============================] - 21s 187ms/step - loss: 0.1836 - accuracy: 0.9557 - val_loss: 0.3588 - val_accuracy: 0.8893
Epoch 4/5
115/115 [==============================] - 20s 175ms/step - loss: 0.1305 - accuracy: 0.9739 - val_loss: 0.3469 - val_accuracy: 0.8934
Epoch 5/5
115/115 [==============================] - 21s 186ms/step - loss: 0.0993 - accuracy: 0.9837 - val_loss: 0.3393 - val_accuracy: 0.8945
Unfreezing base model for fine-tuning...
Epoch 1/10
115/115 [==============================] - 64s 327ms/step - loss: 0.3637 - accuracy: 0

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Success! QAT Model saved as 'mobilenet_v2_pet_qat.tflite'


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from tqdm import tqdm

# --- 1. Load the TFLite Model ---
model_path = "mobilenet_v2_pet_qat.tflite"

interpreter = tf.lite.Interpreter(
    model_path=model_path,
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
)

interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_shape = input_details['shape']
input_dtype = input_details['dtype']

print(f"Model Expected Input Shape: {input_shape}")
print(f"Model Expected Input Dtype: {input_dtype}")

# --- 2. Load Oxford-IIIT Pet Dataset ---
dataset = tfds.load('oxford_iiit_pet', split='test', as_supervised=True)

# --- 3. Preprocessing Logic ---
def preprocess(image, label):
    # Resize
    image = tf.image.resize(image, (input_shape[1], input_shape[2]))

    # Base MobileNetV2 float scaling: map [0, 255] to [-1.0, 1.0]
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0

    if input_dtype == np.int8:
        scale, zero_point = input_details['quantization']
        # Shift the [-1.0, 1.0] float image into the INT8 domain
        image = tf.round(image / scale) + zero_point
        image = tf.clip_by_value(image, -128, 127)
        image = tf.cast(image, tf.int8)

    return image, label

# --- 4. Run Inference ---
correct_predictions = 0
total_samples = 0

print("Running inference on the test set...")
for image, label in tqdm(dataset):
    processed_image, true_label = preprocess(image, label)
    input_data = tf.expand_dims(processed_image, axis=0)

    interpreter.set_tensor(input_details['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details['index'])

    predicted_label = np.argmax(output_data)

    if predicted_label == true_label.numpy():
        correct_predictions += 1
    total_samples += 1

# --- 5. Results ---
accuracy = (correct_predictions / total_samples) * 100
print(f"\nFinal Test Accuracy: {accuracy:.2f}%")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Model Expected Input Shape: [  1 224 224   3]
Model Expected Input Dtype: <class 'numpy.int8'>
Running inference on the test set...


100%|██████████| 3669/3669 [02:16<00:00, 26.91it/s]


Final Test Accuracy: 80.76%
